In [2]:
import pandas as pd
import numpy as np

In [3]:
# keep_default_na=False stops pandas from turning the word "None" into a missing value
df1 = pd.read_csv("../data/raw/Sleep_health_and_lifestyle_dataset.csv",
                  keep_default_na=False, na_values=[""])
df2 = pd.read_csv("../data/raw/sleep-health-lifestyle.csv",
                  keep_default_na=False, na_values=[""])
# we have to ensure the word "None" is not treated as a missing value
for name, df in [("df1", df1), ("df2", df2)]:
    print("=====", name, df.shape)
    print(df.dtypes)
    print(df.head(3))

===== df1 (374, 13)
Person ID                    int64
Gender                         str
Age                          int64
Occupation                     str
Sleep Duration             float64
Quality of Sleep             int64
Physical Activity Level      int64
Stress Level                 int64
BMI Category                   str
Blood Pressure                 str
Heart Rate                   int64
Daily Steps                  int64
Sleep Disorder                 str
dtype: object
   Person ID Gender  Age         Occupation  Sleep Duration  Quality of Sleep  \
0          1   Male   27  Software Engineer             6.1                 6   
1          2   Male   28             Doctor             6.2                 6   
2          3   Male   28             Doctor             6.2                 6   

   Physical Activity Level  Stress Level BMI Category Blood Pressure  \
0                       42             6   Overweight         126/83   
1                       60             8  

In [4]:
def clean_cols(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"[^\w]+", "_", regex=True).str.strip("_"))
    return df

df1, df2 = clean_cols(df1), clean_cols(df2)

print("Only in df1:", sorted(set(df1.columns) - set(df2.columns)))
print("Only in df2:", sorted(set(df2.columns) - set(df1.columns)))
print("In both:   ", sorted(set(df1.columns) & set(df2.columns)))

Only in df1: ['blood_pressure', 'heart_rate', 'occupation', 'person_id', 'physical_activity_level', 'quality_of_sleep', 'sleep_disorder', 'sleep_duration', 'stress_level']
Only in df2: ['alcohol_drinks_per_week', 'health_risk', 'health_risk_score', 'heart_rate_bpm', 'married', 'physical_activity_minutes_per_day', 'quality_of_sleep_1_10', 'sleep_duration_hours', 'smoking', 'stress_level_1_10', 'sugar_intake_g_per_day']
In both:    ['age', 'bmi_category', 'daily_steps', 'gender']


In [5]:
rename_map = {
    "heart_rate_bpm": "heart_rate",
    "quality_of_sleep_1_10": "quality_of_sleep",
    "sleep_duration_hours": "sleep_duration",
    "stress_level_1_10": "stress_level",
    "physical_activity_minutes_per_day": "physical_activity_level",
}
df2 = df2.rename(columns=rename_map)

print("In both:   ", sorted(set(df1.columns) & set(df2.columns)))
print("Only in df1:", sorted(set(df1.columns) - set(df2.columns)))
print("Only in df2:", sorted(set(df2.columns) - set(df1.columns)))

In both:    ['age', 'bmi_category', 'daily_steps', 'gender', 'heart_rate', 'physical_activity_level', 'quality_of_sleep', 'sleep_duration', 'stress_level']
Only in df1: ['blood_pressure', 'occupation', 'person_id', 'sleep_disorder']
Only in df2: ['alcohol_drinks_per_week', 'health_risk', 'health_risk_score', 'married', 'smoking', 'sugar_intake_g_per_day']


In [6]:
for df in (df1, df2):
    for c in df.select_dtypes("object").columns:
        df[c] = df[c].str.strip().str.lower()

common = sorted(set(df1.columns) & set(df2.columns))

# Categorical columns: do both files use the same words?
for c in common:
    if df1[c].dtype == "object":
        a, b = set(df1[c].unique()), set(df2[c].unique())
        if a != b:
            print(f"MISMATCH in '{c}':\n  only df1: {sorted(a-b)[:8]}\n  only df2: {sorted(b-a)[:8]}")

# Numeric columns: same scale in both files?
num_common = [c for c in common if pd.api.types.is_numeric_dtype(df1[c])
              and pd.api.types.is_numeric_dtype(df2[c])]
print(pd.concat({"df1": df1[num_common].agg(["min", "max", "mean"]),
                 "df2": df2[num_common].agg(["min", "max", "mean"])}).round(2))

            age  daily_steps  heart_rate  physical_activity_level  \
df1 min   27.00      3000.00       65.00                    30.00   
    max   59.00     10000.00       86.00                    90.00   
    mean  42.18      6816.84       70.17                    59.17   
df2 min    5.00      1000.00       52.00                     0.00   
    max   80.00     19186.00      200.00                   162.90   
    mean  45.69      6853.41       71.96                    60.92   

          quality_of_sleep  sleep_duration  stress_level  
df1 min               4.00            5.80          3.00  
    max               9.00            8.50          8.00  
    mean              7.31            7.13          5.39  
df2 min               2.20            4.00          1.00  
    max              10.00           15.00         10.00  
    mean              7.03            6.90          4.91  


/tmp/ipykernel_3267/415126555.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes("object").columns:
/tmp/ipykernel_3267/415126555.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on h